In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import glob
from spark_session import get_spark_session

print("Importaciones completas.")

Importaciones completas.


In [2]:
# --- 2. PARÁMETROS DE CONFIGURACIÓN ---

# Ruta LAPTOP (Host) - Para que el Driver lea los archivos
HOST_DATA_PATH = "./../../data" 
# Ruta DENTRO de los contenedores Docker (como en docker-compose.yml)
CONTAINER_DATA_PATH = "/opt/spark/data"

# Ruta base donde se guardarán los resultados (en el Host, para la limpieza inicial)
OUTPUT_HOST_PATH = os.path.join(HOST_DATA_PATH, "processed")
# Ruta base donde los workers escribirán (en el Contenedor)
OUTPUT_CONTAINER_PATH = os.path.join(CONTAINER_DATA_PATH, "processed")

N_SLICES = 16

IMAGE_LIMIT = 1

In [3]:
# --- 3. INICIALIZAR SESIÓN DE SPARK ---
print("Iniciando SparkSession en modo distribuido...")
spark = get_spark_session("TreeRingSlicing (Distributed)")

sc = spark.sparkContext
print("\n--- SparkSession Iniciada --- ✅")
print(f"Spark Version: {sc.version}")
print(f"Master: {sc.master}")
print(f"UI Web: {sc.uiWebUrl}")

Iniciando SparkSession en modo distribuido...
Attempting to connect to master at: spark://localhost:7077
Will announce driver host IP as: 172.26.0.1
Check: IP 172.26.0.1 resolves locally.


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/08 18:11:10 WARN Utils: Your hostname, archlinux, resolves to a loopback address: 127.0.0.1; using 192.168.1.5 instead (on interface wlp0s20f3)
25/11/08 18:11:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/08 18:11:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



--- Connection Successful! --- ✅
SparkSession object: <pyspark.sql.session.SparkSession object at 0x7fef6df4c7f0>
SparkContext object: <SparkContext master=spark://localhost:7077 appName=TreeRingSlicing (Distributed)>
Spark version in use: 4.0.1

--- SparkSession Iniciada --- ✅
Spark Version: 4.0.1
Master: spark://localhost:7077
UI Web: http://172.26.0.1:4040


In [4]:
# --- 4. FUNCIONES AUXILIARES ---

def load_pith_locations(metadata_path):
    """
    Carga el CSV de ubicaciones del centro (pith) en un diccionario
    para una búsqueda rápida.
    """
    try:
        csv_path = os.path.join(metadata_path, 'pith_location.csv')
        df = pd.read_csv(csv_path)
        # Convertimos a un diccionario: {'F10b': (cx, cy)}
        # Nota: El CSV tiene (cy, cx), lo invertimos a (cx, cy) para OpenCV
        pith_dict = {row['Image']: (row['cx'], row['cy']) for index, row in df.iterrows()}
        
        print(f"Cargados {len(pith_dict)} centros (pith) desde CSV.")
        print(f"Ejemplo: {list(pith_dict.items())[0]}")
        return pith_dict
        
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo CSV en {csv_path}")
        return {}
    except Exception as e:
        print(f"Error cargando CSV: {e}")
        return {}

def get_image_files(raw_path, limit=None):
    """
    Escanea la carpeta 'raw' y devuelve una lista de nombres de archivo de imagen.
    """
    try:
        all_files = [f for f in os.listdir(raw_path) if f.lower().endswith('.png')]
        all_files.sort()
        
        if limit is not None:
            print(f"Se encontraron {len(all_files)} imágenes. Procesando las primeras {limit}.")
            return all_files[:limit]
        else:
            print(f"Se encontraron y procesarán {len(all_files)} imágenes.")
            return all_files
            
    except FileNotFoundError:
        print(f"Error: No se encontró el directorio 'raw' en {raw_path}")
        return []

In [ ]:
# --- 5. LÓGICA DE PROCESAMIENTO (PARA LOS WORKERS) ---

def process_image_slice(image_name, base_data_path, output_base_path, n_slices, pith_locations_broadcast):
    """
    Esta es la función principal que se ejecutará en paralelo en cada worker.
    Toma el nombre de una imagen, la carga, la divide en N rebanadas
    y guarda cada rebanada.
    """
    print("algo")
    # --------------------------------------------------
    # 1. Preparar rutas y datos
    # --------------------------------------------------
    try:    
        # Reconstruir rutas (los workers no conocen las variables globales)
        image_path = os.path.join(base_data_path, 'raw', image_name)
        image_name_no_ext = os.path.splitext(image_name)[0]
        
        # Obtener el diccionario de centros (pith) desde la variable transmitida
        pith_map = pith_locations_broadcast.value
        
        # Cargar la imagen
        img = cv2.imread(image_path)
        if img is None:
            return (image_name, "ERROR: No se pudo leer la imagen")
            
        h, w, _ = img.shape
        
        # --------------------------------------------------
        # 2. Encontrar el centro (Pith)
        # --------------------------------------------------
        if image_name_no_ext not in pith_map:
            return (image_name, f"ERROR: No se encontró el centro (pith) para {image_name_no_ext}")
            
        # (cx, cy) - Formato de OpenCV (x, y)
        (cx, cy) = pith_map[image_name_no_ext]
        # --------------------------------------------------
        # 3. Crear y guardar cada "rebanada"
        # --------------------------------------------------
        
        # Calcular el radio máximo para cubrir toda la imagen desde el centro
        radius = int(np.sqrt(max(cx, w - cx)**2 + max(cy, h - cy)**2)) + 1
        
        # Definir el directorio de salida para esta imagen
        output_dir = os.path.join(output_base_path, str(n_slices), image_name_no_ext)
        os.makedirs(output_dir, exist_ok=True)
        
        angle_per_slice = 360.0 / n_slices
        
        slices_saved = 0
        for i in range(n_slices):
            # Calcular los ángulos de inicio y fin para la rebanada
            # Restamos 90 grados porque cv2.ellipse empieza a las 3 en punto,
            # y nosotros queremos empezar a las 12 en punto.
            
            start_angle = (i * angle_per_slice) - 90
            end_angle = ((i + 1) * angle_per_slice) - 90
            
            # 1. Crear una máscara en blanco
            mask = np.zeros((h, w), dtype=np.uint8)
            
            # 2. Dibujar la "rebanada de pastel" (un sector de elipse) en la máscara
            cv2.ellipse(
                mask,
                center=(int(cx), int(cy)),
                axes=(int(radius), int(radius)), # Ejes (radio)
                angle=0,                      # Sin rotación
                startAngle=start_angle,
                endAngle=end_angle,
                color=255,                    # Color blanco
                thickness=-1                  # Relleno
            )
            
            # 3. Aplicar la máscara a la imagen original
            result = cv2.bitwise_and(img, img, mask=mask)
            
            # 4. Guardar el resultado
            output_filename = f"{image_name_no_ext}_{i+1}.png"
            output_path = os.path.join(output_dir, output_filename)
            cv2.imwrite(output_path, result)
            slices_saved += 1
            
        return (image_name, f"SUCCESS: {slices_saved} rebanadas guardadas")

    except Exception as e:
        # Capturar cualquier error inesperado
        return (image_name, f"ERROR: {str(e)}")

In [8]:
# --- 6. SPARK PIPELINE EXECUTION ---

print("=" * 60)
print("Starting Spark Processing Pipeline")
print("=" * 60)

# ------------------------------------------------------------------------------
# STEP 0: Prepare output directory with proper permissions
# ------------------------------------------------------------------------------
print("\n[0/5] Preparing output directory...")

# Create base output directory if it doesn't exist
os.makedirs(OUTPUT_HOST_PATH, exist_ok=True)

# Clean previous output for this specific N_SLICES configuration
output_path_full = os.path.join(OUTPUT_HOST_PATH, str(N_SLICES))
if os.path.exists(output_path_full):
    print(f"  → Cleaning previous output: {output_path_full}")
    shutil.rmtree(output_path_full)

# Recreate with proper permissions for Docker containers
os.makedirs(output_path_full, exist_ok=True)
os.chmod(OUTPUT_HOST_PATH, 0o777)
os.chmod(output_path_full, 0o777)
print(f"  ✓ Output directory ready: {output_path_full}")

# ------------------------------------------------------------------------------
# STEP 1: Load and broadcast metadata (Driver-side)
# ------------------------------------------------------------------------------
print("\n[1/5] Loading pith location metadata (Driver-side)...")

pith_dict = load_pith_locations(os.path.join(HOST_DATA_PATH, 'metadata'))
if not pith_dict:
    raise ValueError("Failed to load pith data. Aborting.")

pith_broadcast = sc.broadcast(pith_dict)
print(f"  ✓ Broadcast {len(pith_dict)} pith locations to workers")
print(f"  Pith dict example: {pith_dict}")
# ------------------------------------------------------------------------------
# STEP 2: Get list of images to process (Driver-side)
# ------------------------------------------------------------------------------
print("\n[2/5] Scanning for image files (Driver-side)...")

image_files = get_image_files(os.path.join(HOST_DATA_PATH, 'raw'), limit=IMAGE_LIMIT)
if not image_files:
    raise ValueError("No images found to process. Aborting.")

print(f"  ✓ Found {len(image_files)} image(s) to process")

# ------------------------------------------------------------------------------
# STEP 3: Create RDD for distributed processing
# ------------------------------------------------------------------------------
print("\n[3/5] Creating RDD for parallel processing...")

image_rdd = sc.parallelize(image_files)
print(f"  ✓ RDD created with {image_rdd.getNumPartitions()} partition(s)")

# ------------------------------------------------------------------------------
# STEP 4: Define worker function and execute transformations
# ------------------------------------------------------------------------------
print("\n[4/5] Executing distributed image slicing...")


def run_slicing(image_name):
    """
    Wrapper function that will run on each Spark worker.
    Passes container paths (not host paths) to the processing function.
    """
    return process_image_slice(
        image_name,
        CONTAINER_DATA_PATH,      # Path inside Docker container
        OUTPUT_CONTAINER_PATH,     # Path inside Docker container
        N_SLICES,
        pith_broadcast
    )

# Execute the transformation (map) and action (collect)
results = image_rdd.map(run_slicing).collect()

print(f"  ✓ Processing completed!")

# ------------------------------------------------------------------------------
# STEP 5: Display execution results
# ------------------------------------------------------------------------------
print("\n[5/5] Execution Summary:")
print("-" * 60)

for image_name, status in results:
    status_icon = "✓" if "SUCCESS" in status else "✗"
    print(f"  {status_icon} {image_name}: {status}")

print("-" * 60)
print("\n✅ Pipeline execution completed successfully!")
print("=" * 60)

Starting Spark Processing Pipeline

[0/5] Preparing output directory...
  ✓ Output directory ready: ./../../data/processed/16

[1/5] Loading pith location metadata (Driver-side)...
Cargados 64 centros (pith) desde CSV.
Ejemplo: ('F10b', (884, 780))
  ✓ Broadcast 64 pith locations to workers
  Pith dict example: {'F10b': (884, 780), 'F10a': (1106, 1146), 'F10e': (850, 967), 'F02c': (1264, 1204), 'F02b': (903, 879), 'F02a': (1197, 1293), 'F02d': (1086, 1200), 'F02e': (1038, 1069), 'F03c': (1208, 1248), 'F03b': (943, 946), 'F03a': (1338, 1311), 'F03d': (1180, 1248), 'F03e': (926, 1060), 'F04c': (472, 466), 'F04b': (1034, 878), 'F04a': (1276, 1206), 'F04d': (454, 412), 'F04e': (1033, 1068), 'F07c': (515, 486), 'F07b': (814, 879), 'F07a': (1186, 1254), 'F07d': (505, 443), 'F07e': (952, 984), 'F08c': (1274, 1399), 'F08b': (954, 893), 'F08a': (1207, 1198), 'F08d': (1218, 1038), 'F08e': (946, 863), 'F09c': (1142, 1178), 'F09b': (925, 961), 'F09a': (1092, 1036), 'F09e': (796, 805), 'L11b': (918

  ✓ Processing completed!

[5/5] Execution Summary:
------------------------------------------------------------
  ✓ F02a.png: SUCCESS: 16 rebanadas guardadas
------------------------------------------------------------

✅ Pipeline execution completed successfully!


In [ ]:
# --- 7. VERIFICACIÓN DE RESULTADOS ---

print(f"Mostrando resultados para la primera imagen procesada: {image_files[0]}")

image_name_no_ext = os.path.splitext(image_files[0])[0]
# Usar la ruta del HOST para encontrar los archivos generados
slice_dir = os.path.join(OUTPUT_HOST_PATH, str(N_SLICES), image_name_no_ext) # <--- FIX #6
slice_files = sorted(glob.glob(os.path.join(slice_dir, "*.png")))

if not slice_files:
    print(f"No se encontraron rebanadas generadas en {slice_dir}")
else:
    # Crear una grilla de 4x4 para mostrar las 16 rebanadas
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    fig.suptitle(f'Rebanadas generadas para {image_name_no_ext} (N={N_SLICES})', fontsize=16)
    
    # Aplanar el array de ejes para una iteración fácil
    axes = axes.flatten()
    
    for i, ax in enumerate(axes):
        if i < len(slice_files):
            # Cargar la rebanada guardada
            slice_img = cv2.imread(slice_files[i])
            slice_img_rgb = cv2.cvtColor(slice_img, cv2.COLOR_BGR2RGB)
            
            ax.imshow(slice_img_rgb)
            ax.set_title(os.path.basename(slice_files[i]), fontsize=8)
            ax.axis('off')
        else:
            # Ocultar ejes si hay menos de 16 rebanadas
            ax.axis('off')
            
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [7]:
# --- 8. DETENER LA SESIÓN DE SPARK ---
print("Deteniendo SparkSession...")
spark.stop()
print("Sesión detenida.")

Deteniendo SparkSession...
Sesión detenida.
